In [ ]:
!pip install statsmodels scikit-learn openpyxl xlrd

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.stattools import durbin_watson
from sklearn.metrics import mean_squared_error, mean_absolute_error
import io
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False
})

print("Setup complete")

 Setup complete


In [ ]:
POLLUTANTS = ["PM2.5", "PM10", "NO2", "O3", "CO", "SO2"]

NAAQS_24HR = {
    "PM2.5": 60,
    "PM10": 100,
    "NO2": 80,
    "O3": 100,
    "CO": 2000,
    "SO2": 80
}

NAAQS_ANNUAL = {
    "PM2.5": 40,
    "PM10": 60,
    "NO2": 40,
    "SO2": 50
}

print("Constants set")

 Constants set


In [ ]:
def load_data(filepath):
    df = pd.read_csv(filepath)
    df.columns = df.columns.str.strip()
    df.rename(columns={
        "Timestamp": "timestamp",
        "date": "timestamp",
        "PM25": "PM2.5",
        "PM_2.5": "PM2.5",
        "PM_10": "PM10"
    }, inplace=True)

    print("Columns found:", df.columns.tolist())

    if "timestamp" not in df.columns:
        raise ValueError("❌ No timestamp column found")

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"])
    df.set_index("timestamp", inplace=True)
    df = df.sort_index()

    for col in POLLUTANTS:
        if col in df.columns:
            df[col] = df[col].clip(lower=0)

    print("Data loaded:", df.shape)
    return df

print("load_data defined")

load_data defined


In [ ]:
def assess_missing_values(df):
    missing = df[POLLUTANTS].isnull().sum()
    pct = (missing / len(df) * 100).round(2)
    print(pd.DataFrame({"Missing": missing, "%": pct}))

def impute_missing(df):
    df = df.copy()
    for col in POLLUTANTS:
        if col in df.columns:
            df[col] = df[col].interpolate(limit=8)
    return df

print(" Missing value functions defined")

 Missing value functions defined


In [ ]:
def extract_time_features(df):
    df = df.copy()
    df["hour"] = df.index.hour
    df["month"] = df.index.month
    df["year"] = df.index.year
    df["day_of_week"] = df.index.dayofweek
    return df

print(" extract_time_features defined")

extract_time_features defined


In [ ]:
def week1(df):
    df = extract_time_features(df)
    print(df[POLLUTANTS].describe())

    df["PM2.5"].plot(figsize=(10, 4), title="PM2.5 Time Series")
    plt.ylabel("µg/m³")
    plt.show()

    valid = df[["PM2.5", "PM10"]].dropna()
    if len(valid) > 10:
        m, b, r, _, _ = stats.linregress(valid["PM2.5"], valid["PM10"])
        print(f"PM2.5 vs PM10 linear relation: r = {r:.2f}")

print(" week1 defined")

 week1 defined


In [ ]:
AQI_BREAKPOINTS = {
    "PM2.5": [
        (0, 30, 0, 50),
        (30, 60, 51, 100),
        (60, 90, 101, 200),
        (90, 120, 201, 300),
        (120, 250, 301, 400),
        (250, 500, 401, 500)
    ]
}

def compute_sub_index(c, bp):
    if pd.isna(c):
        return np.nan
    for lo, hi, ilo, ihi in bp:
        if lo <= c <= hi:
            return ilo + (ihi - ilo) / (hi - lo) * (c - lo)
    return 500

def compute_aqi(df):
    daily = df["PM2.5"].resample("D").mean()
    aqi = daily.apply(lambda x: compute_sub_index(x, AQI_BREAKPOINTS["PM2.5"]))
    return aqi

print("AQI functions defined")

 AQI functions defined


In [ ]:
def week3(df):
    annual = df["PM2.5"].resample("YE").mean()
    print(annual)

    annual.plot(kind="bar", title="Annual Mean PM2.5", color="steelblue")
    plt.ylabel("µg/m³")
    plt.tight_layout()
    plt.show()

print(" week3 defined")

 week3 defined


In [ ]:
def forecasting(df):
    data = df["PM2.5"].resample("D").mean().interpolate().dropna()

    train = data[:-30]
    test  = data[-30:]

    model = SARIMAX(train, order=(2, 1, 2), seasonal_order=(1, 1, 1, 7),
                    enforce_stationarity=False, enforce_invertibility=False)
    fit = model.fit(disp=False)

    pred = fit.forecast(len(test))

    rmse = np.sqrt(mean_squared_error(test, pred))
    mae  = mean_absolute_error(test, pred)
    mape = np.mean(np.abs((test - pred) / (test + 1e-6))) * 100

    print(f"RMSE : {rmse:.2f}")
    print(f"MAE  : {mae:.2f}")
    print(f"MAPE : {mape:.2f}%")

    plt.figure(figsize=(10, 4))
    plt.plot(test.index, test.values,  label="Actual",   color="steelblue")
    plt.plot(pred.index, pred.values,  label="Forecast", color="tomato", linestyle="--")
    plt.legend()
    plt.title("PM2.5 — SARIMA Validation (Last 30 Days)")
    plt.ylabel("µg/m³")
    plt.tight_layout()
    plt.show()

print("forecasting defined")

forecasting defined


In [ ]:
from google.colab import files

print("Upload your historical file: project_airQ.csv")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
print("Uploaded:", filename)

df = pd.read_csv(io.BytesIO(uploaded[filename]))
df.columns = df.columns.str.strip()
df.rename(columns={
    "Timestamp": "timestamp",
    "date": "timestamp",
    "PM25": "PM2.5",
    "PM_2.5": "PM2.5",
    "PM_10": "PM10"
}, inplace=True)

print("Columns:", df.columns.tolist())

if "timestamp" not in df.columns:
    raise ValueError(" No timestamp column — check column names above")

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df.dropna(subset=["timestamp"])
df.set_index("timestamp", inplace=True)
df = df.sort_index()

for col in POLLUTANTS:
    if col in df.columns:
        df[col] = df[col].clip(lower=0)

print(" Historical data loaded:", df.shape)

# Run pipeline
assess_missing_values(df)
df = impute_missing(df)
week1(df)

aqi = compute_aqi(df)
print("\nSample AQI values:")
print(aqi.head())

week3(df)
forecasting(df)

Upload your historical file: project_airQ.csv


In [ ]:
def forecast_future(df, days=365):
    data = df["PM2.5"].resample("D").mean().interpolate().dropna()

    model = SARIMAX(
        data,
        order=(2, 1, 2),
        seasonal_order=(1, 1, 1, 7),
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    fit = model.fit(disp=False)
    forecast = fit.forecast(steps=days)

    future_dates = pd.date_range(
        start=data.index[-1] + pd.Timedelta(days=1),
        periods=days,
        freq="D"
    )
    forecast.index = future_dates

    plt.figure(figsize=(12, 5))
    plt.plot(data[-60:], label="Last 60 days (actual)", color="steelblue")
    plt.plot(forecast, label=f"Next {days} days (forecast)",
             color="tomato", linestyle="--")
    plt.axhline(y=60, color="gray", linestyle=":", linewidth=1, label="NAAQS limit")
    plt.legend()
    plt.title(f"PM2.5 Future Forecast — Next {days} Days")
    plt.ylabel("µg/m³")
    plt.tight_layout()
    plt.show()

    return forecast

# Call it
future = forecast_future(df, days=365)

print("Future forecast complete")

In [ ]:
from google.colab import files

print("Upload: Bhopal_AQ_2025_Cleaned.xlsx")
uploaded_2025 = files.upload()

filename_2025 = list(uploaded_2025.keys())[0]
print("Uploaded:", filename_2025)

if not filename_2025.endswith('.xlsx'):
    raise ValueError(f" Wrong file! Got '{filename_2025}' — upload the .xlsx file")

df_2025 = pd.read_excel(
    io.BytesIO(uploaded_2025[filename_2025]),
    sheet_name='For_Notebook (flat)',
    skiprows=1,
    engine='openpyxl'
)

df_2025.columns = df_2025.columns.str.strip()
df_2025.rename(columns={
    'O3 (Ozone)'      : 'O3',
    'WS (wind speed)' : 'WS',
    'WD (wind dir)'   : 'WD'
}, inplace=True)

df_2025['timestamp'] = pd.to_datetime(df_2025['timestamp'], errors='coerce')
df_2025 = df_2025.dropna(subset=['timestamp'])
df_2025.set_index('timestamp', inplace=True)
df_2025 = df_2025.sort_index()

for col in df_2025.columns:
    df_2025[col] = pd.to_numeric(df_2025[col], errors='coerce').clip(lower=0)

print("2025 data loaded:", df_2025.shape)
print(df_2025.head(3))

In [ ]:
def compare_forecast_vs_actual(df_historical, df_2025, days=365):

    # Train on full historical data
    train_data = df_historical["PM2.5"].resample("D").mean().interpolate().dropna()

    model = SARIMAX(train_data, order=(2, 1, 2), seasonal_order=(1, 1, 1, 7),
                    enforce_stationarity=False, enforce_invertibility=False)
    fit = model.fit(disp=False)

    # Forecast 365 days
    forecast = fit.forecast(steps=days)
    future_dates = pd.date_range(
        start=train_data.index[-1] + pd.Timedelta(days=1),
        periods=days, freq="D"
    )
    forecast.index = future_dates

    # Align with actual 2025
    actual_2025 = df_2025["PM2.5"].resample("D").mean().interpolate().dropna()
    common_dates = forecast.index.intersection(actual_2025.index)

    if len(common_dates) == 0:
        print(" No overlapping dates between forecast and 2025 data.")
        print("   Forecast range:", forecast.index[0], "→", forecast.index[-1])
        print("   Actual range:  ", actual_2025.index[0], "→", actual_2025.index[-1])
        return

    forecast_aligned = forecast[common_dates]
    actual_aligned   = actual_2025[common_dates]

    # Metrics
    rmse = np.sqrt(mean_squared_error(actual_aligned, forecast_aligned))
    mae  = mean_absolute_error(actual_aligned, forecast_aligned)
    mape = np.mean(np.abs((actual_aligned - forecast_aligned) / (actual_aligned + 1e-6))) * 100

    print("=== Forecast vs Actual 2025 ===")
    print(f"Overlapping days : {len(common_dates)}")
    print(f"RMSE             : {rmse:.2f} µg/m³")
    print(f"MAE              : {mae:.2f} µg/m³")
    print(f"MAPE             : {mape:.2f}%")

    # Plot 1 — time series comparison
    plt.figure(figsize=(14, 5))
    plt.plot(actual_aligned,   label="Actual 2025",      color="steelblue", linewidth=1.5)
    plt.plot(forecast_aligned, label="SARIMA Forecast",  color="tomato", linestyle="--", linewidth=1.5)
    plt.fill_between(common_dates,
                     forecast_aligned * 0.85,
                     forecast_aligned * 1.15,
                     alpha=0.15, color="tomato", label="±15% uncertainty band")
    plt.axhline(y=60, color="gray", linestyle=":", linewidth=1, label="NAAQS 24hr limit")
    plt.title("PM2.5 — SARIMA Forecast vs Actual 2025")
    plt.ylabel("µg/m³")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Plot 2 — monthly breakdown
    comparison = pd.DataFrame({"Actual": actual_aligned, "Forecast": forecast_aligned})
    comparison["month"] = comparison.index.month
    monthly = comparison.groupby("month")[["Actual", "Forecast"]].mean()

    monthly.plot(kind="bar", figsize=(10, 4),
                 color=["steelblue", "tomato"],
                 title="Monthly Average — Forecast vs Actual 2025")
    plt.ylabel("PM2.5 (µg/m³)")
    plt.xticks(ticks=range(12),
               labels=["Jan","Feb","Mar","Apr","May","Jun",
                       "Jul","Aug","Sep","Oct","Nov","Dec"],
               rotation=45)
    plt.tight_layout()
    plt.show()

    # Plot 3 — residuals
    residuals = actual_aligned - forecast_aligned
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    residuals.plot(ax=axes[0], color="gray", title="Residuals (Actual − Forecast)")
    axes[0].axhline(0, color="red", linestyle="--")
    axes[0].set_ylabel("µg/m³")
    residuals.plot(kind="hist", ax=axes[1], bins=30,
                   color="steelblue", title="Residual Distribution")
    plt.tight_layout()
    plt.show()

    print(f"Mean residual : {residuals.mean():.2f} µg/m³")
    print(f"Std residuals : {residuals.std():.2f} µg/m³")

    return comparison

# Run it
comparison = compare_forecast_vs_actual(df, df_2025)